# Week 4 — Evaluation + Robustness Stress Tests

**Project:** Can We Trust AI-Image Detectors on Ugandan Content?
**Member:** AW
**Goal:** Evaluate the best model from Week 3 on the test set, then stress-test with resize, JPEG compression, blur, and noise.

### RAM-safe design
- Loads saved model from `reports/`
- 128×128 images, batch 8
- Stress tests applied on-the-fly
- Run cells one at a time

### Drive paths
- Root: `/content/drive/MyDrive/Deepfake Detection Project/datasets`
- Outputs: `reports/week4_robustness_results.json`

In [ ]:
!pip -q install torch torchvision scikit-learn pandas pillow

In [ ]:
import os, sys, json, glob, gc
from datetime import datetime
import numpy as np
import pandas as pd
from PIL import Image as PILImage, ImageFilter, ImageOps
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print('PyTorch:', torch.__version__)
print('Imports OK')

## 1. Mount Drive + Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
ROOT = '/content/drive/MyDrive/Deepfake Detection Project/datasets'
MEMBER = 'AW'
IMG_SIZE = 128
BATCH_SIZE = 8
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
for d in ['reports', 'metadata', 'audit_outputs']:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)
print('Drive root:', ROOT)

## 2. Load public + Kampala test images
Uses the same split logic as Week 3: public images for train/val, Kampala as held-out test.

In [ ]:
real_dir = os.path.join(ROOT, 'pilot_data', 'real')
ai_dir = os.path.join(ROOT, 'pilot_data', 'ai')

def load_public_images(real_dir, ai_dir, n_each, img_size):
    paths, labels, meta = [], [], []
    real_files = sorted(glob.glob(os.path.join(real_dir, '*.jpg'))) + sorted(glob.glob(os.path.join(real_dir, '*.png')))
    for p in real_files[:n_each]:
        paths.append(p); labels.append(0)
        meta.append({'path': p, 'label': 'real', 'source': 'pilot'})
    ai_files = sorted(glob.glob(os.path.join(ai_dir, '*.jpg'))) + sorted(glob.glob(os.path.join(ai_dir, '*.png')))
    for p in ai_files[:n_each]:
        paths.append(p); labels.append(1)
        meta.append({'path': p, 'label': 'ai', 'source': 'pilot'})
    if len(paths) == 0:
        print('No pilot images — creating synthetic set.')
        for i in range(n_each):
            arr = np.random.randint(0, 255, (img_size, img_size, 3), dtype=np.uint8)
            im = PILImage.fromarray(arr)
            p = os.path.join(real_dir, f'synthetic_real_{i:03d}.jpg')
            im.save(p, quality=95)
            paths.append(p); labels.append(0)
            meta.append({'path': p, 'label': 'real', 'source': 'synthetic'})
            arr = np.zeros((img_size, img_size, 3), dtype=np.uint8)
            for x in range(img_size):
                arr[:, x, :] = [int(255 * x / img_size), 128, 64]
            im = PILImage.fromarray(arr)
            p = os.path.join(ai_dir, f'synthetic_ai_{i:03d}.jpg')
            im.save(p, quality=95)
            paths.append(p); labels.append(1)
            meta.append({'path': p, 'label': 'ai', 'source': 'synthetic'})
    return paths, labels, meta

n_each = 100
public_paths, public_labels, public_meta = load_public_images(real_dir, ai_dir, n_each, IMG_SIZE)

kampala_dirs = {
    'orig': os.path.join(ROOT, 'kampala', 'originals'),
    'wa_photo': os.path.join(ROOT, 'kampala', 'whatsapp_photo'),
    'wa_doc': os.path.join(ROOT, 'kampala', 'whatsapp_document'),
}
kampala_log_path = os.path.join(ROOT, 'metadata', 'kampala_log.csv')
kampala_paths, kampala_labels, kampala_meta = [], [], []
if os.path.exists(kampala_log_path):
    log = pd.read_csv(kampala_log_path)
    if len(log) > 0:
        ids = log['image_id'].unique()[:3]
        for image_id in ids:
            rows = log[log['image_id'] == image_id]
            label = 0 if rows['label'].iloc[0] == 'real' else 1
            for _, row in rows.iterrows():
                folder = kampala_dirs.get(row['version'])
                if folder and os.path.exists(os.path.join(folder, str(row['file_name']))):
                    kampala_paths.append(os.path.join(folder, str(row['file_name'])))
                    kampala_labels.append(label)
                    kampala_meta.append({'path': '', 'label': rows['label'].iloc[0], 'source': 'kampala', 'image_id': image_id, 'version': row['version']})

all_paths = public_paths + kampala_paths
all_labels = public_labels + kampala_labels
all_meta = public_meta + kampala_meta
print(f'Total: {len(all_paths)} (public={len(public_paths)}, kampala={len(kampala_paths)})')
print(f'Labels: real={sum(1 for l in all_labels if l==0)}, ai={sum(1 for l in all_labels if l==1)}')

## 3. Tensors + split

In [ ]:
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def images_to_tensor(paths, labels, img_size):
    X_list, y_list = [], []
    for p, l in zip(paths, labels):
        im = PILImage.open(p).convert('RGB')
        t = normalize(to_tensor(im))
        X_list.append(t)
        y_list.append(float(l))
        im.close()
    return torch.stack(X_list), torch.tensor(y_list, dtype=torch.float32)

X_all, y_all = images_to_tensor(all_paths, all_labels, IMG_SIZE)

is_kampala = np.array([m['source'] == 'kampala' for m in all_meta])
public_idx = np.where(~is_kampala)[0]
kampala_idx = np.where(is_kampala)[0]

from sklearn.model_selection import train_test_split
if len(public_idx) > 0:
    pub_X = X_all[public_idx]
    pub_y = y_all[public_idx]
    pub_meta = [all_meta[i] for i in public_idx]
    X_tr, X_te, y_tr, y_te, meta_tr, meta_te = train_test_split(pub_X, pub_y, pub_meta, test_size=0.2, random_state=SEED, stratify=pub_y.int().numpy())
else:
    X_tr, X_te, y_tr, y_te, meta_tr, meta_te = X_all[:0], X_all[:0], y_all[:0], y_all[:0], [], []

if len(kampala_idx) > 0:
    X_kamp = X_all[kampala_idx]
    y_kamp = y_all[kampala_idx]
    meta_kamp = [all_meta[i] for i in kampala_idx]
else:
    X_kamp, y_kamp, meta_kamp = X_all[:0], y_all[:0], []

test_ds = TensorDataset(X_te, y_te)
kampala_ds = TensorDataset(X_kamp, y_kamp)

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
kampala_loader = DataLoader(kampala_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

print(f'test={len(test_ds)} kampala={len(kampala_ds)}')
print(f'test labels: real={int((y_te==0).sum())} ai={int((y_te==1).sum())}')

## 4. TinyCNN model + load best checkpoint

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, img_size=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x.squeeze(1)

model = TinyCNN(img_size=IMG_SIZE)
model_path = os.path.join(ROOT, 'reports', 'week3_run_c_aug.pt')
if not os.path.exists(model_path):
    model_path = os.path.join(ROOT, 'reports', 'tinycnn_baseline.pt')
    print('Falling back to Week 2 baseline')

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    print(f'Loaded {model_path}')
else:
    print('WARNING: No saved model found — using untrained model')

model = model.to(DEVICE)
model.eval()

## 5. Stress test functions
Applies transformations to test images and measures performance drop.

In [ ]:
def apply_stress(img_tensor, stress_type, severity=1.0):
    results = []
    for i in range(img_tensor.size(0)):
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = img_tensor[i] * std + mean
        img = torch.clamp(img, 0, 1)
        pil = transforms.ToPILImage()(img.cpu())
        
        if stress_type == 'resize_small':
            w, h = pil.size
            pil = pil.resize((max(16, int(w * 0.5)), max(16, int(h * 0.5))))
            pil = pil.resize((w, h))
        elif stress_type == 'resize_large':
            w, h = pil.size
            pil = pil.resize((min(512, int(w * 1.5)), min(512, int(h * 1.5))))
            pil = pil.resize((w, h))
        elif stress_type == 'jpeg':
            import io
            buf = io.BytesIO()
            quality = max(10, int(95 - 30 * severity))
            pil.save(buf, format='JPEG', quality=quality)
            buf.seek(0)
            pil = PILImage.open(buf).convert('RGB')
        elif stress_type == 'blur':
            radius = max(1, int(2 * severity))
            pil = pil.filter(ImageFilter.GaussianBlur(radius=radius))
        elif stress_type == 'noise':
            arr = np.array(pil).astype(np.float32)
            noise = np.random.normal(0, 25 * severity, arr.shape)
            arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
            pil = PILImage.fromarray(arr)
        elif stress_type == 'brightness':
            factor = 1.0 + 0.5 * severity
            pil = ImageOps.autocontrast(pil)
            arr = np.array(pil).astype(np.float32) * factor
            arr = np.clip(arr, 0, 255).astype(np.uint8)
            pil = PILImage.fromarray(arr)
        
        t = transforms.ToTensor()(pil)
        t = normalize(t)
        results.append(t)
    return torch.stack(results)

def evaluate_with_stress(model, loader, stress_type=None, severity=1.0, device=DEVICE):
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            if stress_type is not None:
                xb = apply_stress(xb, stress_type, severity)
                xb = xb.to(device)
            probs = torch.sigmoid(model(xb)).cpu().numpy()
            y_prob.extend(probs.tolist())
            y_true.extend(yb.numpy().tolist())
    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred) if len(y_true) > 0 else 0.0
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float('nan')
    return {'accuracy': float(acc), 'macro_f1': float(f1), 'precision': float(prec), 'recall': float(rec), 'roc_auc': float(auc), 'n': int(len(y_true))}

## 6. Run baseline + stress tests

In [ ]:
results = {}

print('Running baseline (no stress)...')
results['baseline'] = evaluate_with_stress(model, test_loader)
print(f'  Baseline test: acc={results["baseline"]["accuracy"]:.4f} macro-F1={results["baseline"]["macro_f1"]:.4f}')

stress_types = ['resize_small', 'resize_large', 'jpeg', 'blur', 'noise', 'brightness']
for st in stress_types:
    print(f'Running stress: {st}...')
    res = evaluate_with_stress(model, test_loader, stress_type=st, severity=1.0)
    results[st] = res
    print(f'  {st}: acc={res["accuracy"]:.4f} macro-F1={res["macro_f1"]:.4f}')
    gc.collect()

if len(kampala_ds) > 0:
    print('\nKampala held-out evaluation:')
    results['kampala_heldout'] = evaluate_with_stress(model, kampala_loader)
    print(f'  Kampala: acc={results["kampala_heldout"]["accuracy"]:.4f} macro-F1={results["kampala_heldout"]["macro_f1"]:.4f} (n={results["kampala_heldout"]["n"]})')

output = {
    'week': 4,
    'member': MEMBER,
    'model': model_path,
    'img_size': IMG_SIZE,
    'results': results,
    'timestamp': datetime.utcnow().isoformat() + 'Z',
}
out_path = os.path.join(ROOT, 'reports', 'week4_robustness_results.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'\n=== WEEK 4 SUMMARY ===')
print(f'Baseline test macro-F1: {results["baseline"]["macro_f1"]:.4f}')
for st in stress_types:
    print(f'{st:15s}: macro-F1={results[st]["macro_f1"]:.4f} (drop={results["baseline"]["macro_f1"] - results[st]["macro_f1"]:.4f})')
if len(kampala_ds) > 0:
    print(f'Kampala held-out macro-F1: {results["kampala_heldout"]["macro_f1"]:.4f}')
print(f'\nSaved to {out_path}')
gc.collect()